first we need to install selenium.
so copy paste `pip install selenium` and execute it as following:
1. open powershel and then `python pip install selenium`
2. or you can install it on this project level so you just need to add `pip` to this project:
   1. open a new terminal (VS-code or Cursor)
   2. execute `python -m ensurepip --upgrade`
   3. now run `python pip install selenium`

below we are writing python code to read a site and wait to be fully rendered then we return the results as html string

In [ ]:

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

def readClientRenderedHTML(url):
    # Chrome options
    options = Options()
    #options.add_argument("--headless")  # run without opening browser

    # Start browser
    driver = webdriver.Chrome(options=options)

    # Open page
    driver.get(url)

    # Wait for JavaScript to render
    time.sleep(3)

    # Get rendered HTML
    html = driver.page_source


    # Close browser
    driver.quit()

    return html

now we test our new function, just run the code below

In [ ]:
result  = readClientRenderedHTML("https://khaldoon-saqallah.com")
print(result)

now we need to use that function to ask AI about the page content as we did in Week 1 - Day 1 

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

openai = OpenAI(api_key=api_key)

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + "\n\n" + website}
    ]


def summarize(url):
    website = readClientRenderedHTML(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

 

now we execute (invoke) `display_summary` function that we defined above

In [ ]:
display_summary("https://khaldoon-saqallah.com")